**Working on the data cleaning and analysis starts with:
1. Original working csv is: context_barrier_mention_all.csv
2. recovering some rows which the script parser could not write them properly. 
    - Code cell: #cell 2: recovering
    - input:  context_barrier_mention_all.csv
    - A new csv is written: context_barrier_mention_all_recovered.csv
3. Cleaning of malformed generation using: analysis/clean_barrier.py
    - new outputs: 
        - new_clean: analysis/context_barrier_mention_clean.csv
        - report on cleaned: analysis/context_barrier_mention_repetition_report.csv
4. Assigning themes by clustering: named #cell 3 in this notebook:
    - input: context_barrier_mention_clean.csv
    - output: analysis/context_barrier_mention_hdbscan.csv
5. Cluster naming:
    - input: context_barrier_mention_hdbscan.csv
    - GPT naming, output file: context_barrier_mention_hdbscan_gpt5named.csv
    - Human inspection and re-naming of the GPT version, output file: context_barrier_mention_hdbscan_gpt5named_human.csv

6. Verification inspection report on all rows being filled intact:  # cell 5 in this notebook, the outputs are not for exports just views in the notebook

7. A latex table contains the information like (cluster_id, cluster_size, share_of_rows,gpt_cluster_name, human_labelled), all these are in the online appendix attachment; here computed in #cell 6

8. Theme-subthemes mappings for the actual paper contents: 
    - #cell 7a: contains latex exportable table
    - #cell 7b: the Sankey plot on the theme-subthemes mappings

9. Jaccard similarity index: #cell 8: comparing similar role variants for within-model and between models on similar-role variants


In [ ]:
# cell1: detect which variant at what iterations has fewer than 5 rows in the csv.
from pathlib import Path
import pandas as pd

csv_path = Path(r"C:\Users\FEEL\Downloads\images\rag_pipeline\roles-enhanced\expert_responses\context_barrier_mention_all.csv")

df = pd.read_csv(csv_path)

summary = (
    df.groupby(["base_model", "variant_id", "iteration"], as_index=False)
      .size()
      .rename(columns={"size": "row_count"})
      .sort_values(["base_model", "variant_id", "iteration"])
)

few = summary[summary["row_count"] < 5].copy()

print("Groups with fewer than 5 rows:")
display(few)

print("\nCount of problematic groups:", len(few))


Groups with fewer than 5 rows:


,base_model,variant_id,iteration,row_count
62,llama-pro:latest,llama-generalist,2,2
70,llama-pro:latest,llama-generalist2,0,2



Count of problematic groups: 2


In [ ]:
#cell 2: recovering missing barriers by the parser
from pathlib import Path
import pandas as pd

src_csv = Path(r"C:\Users\FEEL\Downloads\images\rag_pipeline\roles-enhanced\expert_responses\context_barrier_mention_all.csv")
out_csv = Path(r"C:\Users\FEEL\Downloads\images\rag_pipeline\roles-enhanced\expert_responses\context_barrier_mention_all_recovered.csv")

df = pd.read_csv(src_csv)

# Full replacement sets for the problematic groups.
# Replace these with the 5 barriers you want to keep from the raw logs.
recovered_full = {
    ("llama-generalist2", 0): [
        {
            "barrier_title": "Legacy Infrastructure",
            "explanation": "The widespread adoption of DC is hindered by the presence of legacy infrastructure, such as electrical systems designed for AC power. This makes it challenging to replace existing systems with DC equivalents without incurring significant costs and disruptions."
        },
        {
            "barrier_title": "Inadequate Regulations",
            "explanation": "The lack of comprehensive regulations and standards surrounding DC technologies can create uncertainty for stakeholders, discouraging investment in new infrastructure and limiting the industry's growth potential."
        },
        {
            "barrier_title": "Lack of Awareness",
            "explanation": "Limited public awareness about the benefits and capabilities of DC systems can lead to resistance from end-users and stakeholders, hindering widespread adoption in consumer sectors like buildings and data centers."
        },
        {
            "barrier_title": "High Implementation Costs",
            "explanation": "DC systems often require specialized hardware components and may necessitate retrofitting existing infrastructure. This can result in significant upfront costs for implementation, making it difficult to justify the investment for smaller-scale applications or projects."
        },
        {
            "barrier_title": "Technical Complexities",
            "explanation": "The complexity of DC systems requires specialized knowledge, skills, and expertise to design, install, operate, and maintain them effectively. This can be a barrier for smaller players or organizations with limited resources and experience in the sector."
        },
    ],
    ("llama-generalist", 2): [
        {
            "barrier_title": "**Limited Understanding of Benefits**",
            "explanation": " Lack of awareness among stakeholders about the advantages of DC systems, leading to resistance in adopting new technologies.",
        },
        {
            "barrier_title": "**High Initial Investment Costs**",
            "explanation": " Compared to traditional AC systems, DC infrastructure requires a significant initial investment, making it less attractive for smaller businesses or projects with limited budgets.",
        },
        {
            "barrier_title": "**Complex Integration with Existing Systems**",
            "explanation": "DC systems often require significant modifications to existing electrical grids and infrastructure, creating compatibility issues and increasing implementation costs.",
        },
        {
            "barrier_title": "**Inadequate Regulatory Guidelines**",
            "explanation": "Lack of clear regulatory frameworks for DC adoption in various industries can cause confusion and hesitation among potential users, leading to slower adoption rates.",
        },
        {
            "barrier_title": "**Technical Challenges in Maintenance and Repair**",
            "explanation": "DC systems may present unique challenges during maintenance and repair due to the complexity of the system components, potentially discouraging businesses from switching to DC technology.",
        }
        ],

}

def make_row(template_row, variant_id, iteration, barrier_index, title, explanation):
    row = template_row.copy()
    row["row_id"] = f"{variant_id}_{int(iteration):02d}_b{barrier_index}"
    row["variant_id"] = variant_id
    row["iteration"] = int(iteration)
    row["barrier_title"] = title
    row["explanation"] = explanation
    return row

# Drop the existing rows for the groups you want to fully replace
mask_drop = pd.Series(False, index=df.index)
for variant_id, iteration in recovered_full.keys():
    mask_drop |= ((df["variant_id"] == variant_id) & (df["iteration"] == iteration))

df_kept = df.loc[~mask_drop].copy()

# Rebuild the full 5-row groups
replacement_rows = []

for (variant_id, iteration), barriers in recovered_full.items():
    original_group = df[(df["variant_id"] == variant_id) & (df["iteration"] == iteration)].copy()
    if original_group.empty:
        raise ValueError(f"No template rows found for {(variant_id, iteration)}")

    template_row = original_group.iloc[0].to_dict()

    if len(barriers) != 5:
        raise ValueError(f"{(variant_id, iteration)} must have exactly 5 recovered barriers")

    for barrier_index, barrier in enumerate(barriers, start=1):
        replacement_rows.append(
            make_row(
                template_row=template_row,
                variant_id=variant_id,
                iteration=iteration,
                barrier_index=barrier_index,
                title=barrier["barrier_title"],
                explanation=barrier["explanation"],
            )
        )

df_replacements = pd.DataFrame(replacement_rows)

combined = pd.concat([df_kept, df_replacements], ignore_index=True)
combined = combined.sort_values(["iteration", "variant_id", "row_id"]).reset_index(drop=True)

combined.to_csv(out_csv, index=False)

print(f"Original CSV unchanged: {src_csv}")
print(f"Recovered CSV written to: {out_csv}")
print(f"Rows removed: {mask_drop.sum()}")
print(f"Rows inserted: {len(df_replacements)}")
print(f"Original rows: {len(df)}")
print(f"New total rows: {len(combined)}")


Original CSV unchanged: C:\Users\FEEL\Downloads\images\rag_pipeline\roles-enhanced\expert_responses\context_barrier_mention_all.csv
Recovered CSV written to: C:\Users\FEEL\Downloads\images\rag_pipeline\roles-enhanced\expert_responses\context_barrier_mention_all_recovered.csv
Rows removed: 4
Rows inserted: 10
Original rows: 1194
New total rows: 1200


In [ ]:
# cell 3: clustering using title+explanation, on: embedding, UMAP, HDBSCAN
from pathlib import Path

import pandas as pd
from sentence_transformers import SentenceTransformer
import umap
import hdbscan

# -----------------------------
# Load data
# -----------------------------
csv_path = Path("analysis/context_barrier_mention_clean.csv")
df = pd.read_csv(csv_path)

df["cluster_text"] = (
    df["barrier_title"].fillna("").astype(str).str.strip()
    + ". "
    + df["explanation"].fillna("").astype(str).str.strip()
)

texts = df["cluster_text"].tolist()

# -----------------------------
# Embeddings
# -----------------------------
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embed_model.encode(
    texts,
    show_progress_bar=True,
    normalize_embeddings=True
)

# -----------------------------
# Optional dimensionality reduction
# -----------------------------
reducer = umap.UMAP(
    n_neighbors=15,
    n_components=10,
    min_dist=0.0,
    metric="cosine",
    random_state=42,
)
X = reducer.fit_transform(embeddings)

# -----------------------------
# HDBSCAN clustering
# -----------------------------
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=10,
    min_samples=5,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True,
)
labels = clusterer.fit_predict(X)

df["cluster"] = labels
df["cluster_prob"] = clusterer.probabilities_

# -----------------------------
# Basic summary
# -----------------------------
n_noise = (df["cluster"] == -1).sum()
n_clusters = df.loc[df["cluster"] >= 0, "cluster"].nunique()

print("Rows:", len(df))
print("Clusters found:", n_clusters)
print("Noise rows (-1):", n_noise)

cluster_summary = (
    df.groupby("cluster")
    .agg(
        rows=("row_id", "count"),
        roles=("role", lambda s: ", ".join(sorted(set(s)))),
        variants=("variant_id", "nunique"),
    )
    .reset_index()
    .sort_values("rows", ascending=False)
)

print(cluster_summary)

# -----------------------------
# Example rows per cluster
# -----------------------------
examples = (
    df.loc[df["cluster"] >= 0, ["cluster", "row_id", "role", "barrier_title"]]
    .sort_values(["cluster", "row_id"])
    .groupby("cluster")
    .head(5)
)

print(examples)

# -----------------------------
# Save output
# -----------------------------
out_path = Path("analysis/context_barrier_mention_hdbscan.csv")
df.to_csv(out_path, index=False)
print("Saved:", out_path)


In [ ]:
#cell 5: verification on all labelled gpt, human, that required are non-empty
from pathlib import Path
import pandas as pd

path = Path(r"C:\Users\FEEL\Downloads\images\rag_pipeline\roles-enhanced\analysis\context_barrier_mention_hdbscan_gpt5named_human.csv")
df = pd.read_csv(path)

df["cluster_name"] = df["cluster_name"].fillna("").astype(str).str.strip()
df["human_labelled"] = df["human_labelled"].fillna("").astype(str).str.strip()

summary = {
    "non_noise_clusters": int(df.loc[df["cluster"] >= 0, "cluster"].nunique()),
    "noise_rows": int((df["cluster"] == -1).sum()),
    "non_noise_rows_missing_gpt_label": int(((df["cluster"] >= 0) & (df["cluster_name"] == "")).sum()),
    "non_noise_rows_missing_human_label": int(((df["cluster"] >= 0) & (df["human_labelled"] == "")).sum()),
    "noise_rows_blank_human_label": int(((df["cluster"] == -1) & (df["human_labelled"] == "")).sum()),
    "noise_rows_nonblank_human_label": int(((df["cluster"] == -1) & (df["human_labelled"] != "")).sum()),
}

summary

In [ ]:
#cell 6: summary of barriers by cluster, with GPT and human labels; an exportable latex table
from pathlib import Path
import pandas as pd

path = Path(r"C:\Users\FEEL\Downloads\images\rag_pipeline\roles-enhanced\analysis\context_barrier_mention_hdbscan_gpt5named_human.csv")
df = pd.read_csv(path)

# Clean label fields
df["cluster_name"] = df["cluster_name"].fillna("").astype(str).str.strip()
df["human_labelled"] = df["human_labelled"].fillna("").astype(str).str.strip()

# Keep only non-noise clusters
work = df[df["cluster"] >= 0].copy()

cluster_table = (
    work.groupby("cluster", as_index=False)
    .agg(
        cluster_size=("row_id", "count"),
        gpt_cluster_name=("cluster_name", lambda s: sorted(set(x for x in s if x))[0] if any(s.fillna("").astype(str).str.strip()) else ""),
        human_labelled=("human_labelled", lambda s: sorted(set(x for x in s if x))[0] if any(s.fillna("").astype(str).str.strip()) else ""),
    )
    .rename(columns={"cluster": "cluster_id"})
    .sort_values(["cluster_size", "cluster_id"], ascending=[False, True])
    .reset_index(drop=True)
)

cluster_table = cluster_table[
    ["cluster_id", "cluster_size", "gpt_cluster_name", "human_labelled"]
]

display(cluster_table)

latex_path = Path(r"C:\Users\FEEL\Downloads\images\rag_pipeline\roles-enhanced\analysis\cluster_id_size_name_table.tex")

latex_str = cluster_table.to_latex(
    index=False,
    escape=True,
    caption="Cluster-level summary of HDBSCAN results with GPT-generated and human-assigned labels.",
    label="tab:cluster_id_size_name",
)

latex_path.write_text(latex_str, encoding="utf-8")

print(f"Saved LaTeX table to: {latex_path}")


In [ ]:
#cell 7a: theme-subtheme-level latex table

from pathlib import Path
import pandas as pd

path = Path(r"C:\Users\FEEL\Downloads\images\rag_pipeline\roles-enhanced\analysis\context_barrier_mention_hdbscan_gpt5named_human.csv")
df = pd.read_csv(path)

# Keep only non-noise rows
df["cluster"] = pd.to_numeric(df["cluster"], errors="coerce")
df = df[df["cluster"] >= 0].copy()

# Clean labels
df["human_labelled"] = df["human_labelled"].fillna("").astype(str).str.strip()
df["cluster_name"] = df["cluster_name"].fillna("").astype(str).str.strip()

# One row per unique cluster-theme-subtheme mapping
cluster_level = (
    df.groupby(["cluster", "human_labelled", "cluster_name"], as_index=False)
      .agg(cluster_size=("row_id", "count"))
)

cluster_level = cluster_level[
    (cluster_level["human_labelled"] != "") &
    (cluster_level["cluster_name"] != "")
].copy()

# Build Theme -> Sub-themes table
theme_table = (
    cluster_level.sort_values(["human_labelled", "cluster_size"], ascending=[True, False])
    .groupby("human_labelled")["cluster_name"]
    .apply(lambda s: "; ".join(pd.unique(s)))
    .reset_index()
    .rename(columns={
        "human_labelled": "Theme",
        "cluster_name": "Sub-themes"
    })
)

display(theme_table)

latex_path = Path(r"C:\Users\FEEL\Downloads\images\rag_pipeline\roles-enhanced\analysis\theme_subthemes_table.tex")

latex_str = theme_table.to_latex(
    index=False,
    escape=True,
    caption="Merged themes and their constituent sub-themes.",
    label="tab:theme_subthemes",
    longtable=False,
)

latex_path.write_text(latex_str, encoding="utf-8")
print(f"Saved LaTeX table to: {latex_path}")


In [ ]:
#cell 7 again on sankey drawing

from pathlib import Path
import pandas as pd
import plotly.graph_objects as go

path = Path(r"C:\Users\FEEL\Downloads\images\rag_pipeline\roles-enhanced\analysis\context_barrier_mention_hdbscan_gpt5named_human.csv")
df = pd.read_csv(path)

# Keep only non-noise rows
df["cluster"] = pd.to_numeric(df["cluster"], errors="coerce")
df = df[df["cluster"] >= 0].copy()

# Clean labels
df["human_labelled"] = df["human_labelled"].fillna("").astype(str).str.strip()
df["cluster_name"] = df["cluster_name"].fillna("").astype(str).str.strip()

# Build one row per unique machine cluster under each human theme
cluster_level = (
    df.groupby(["cluster", "human_labelled", "cluster_name"], as_index=False)
      .agg(cluster_size=("row_id", "count"))
)

# Optional: drop blanks if any slipped through
cluster_level = cluster_level[
    (cluster_level["human_labelled"] != "") &
    (cluster_level["cluster_name"] != "")
].copy()

# Build Sankey nodes
themes = sorted(cluster_level["human_labelled"].unique())
subthemes = sorted(cluster_level["cluster_name"].unique())

theme_nodes = themes
subtheme_nodes = subthemes
all_nodes = theme_nodes + subtheme_nodes

node_index = {name: i for i, name in enumerate(all_nodes)}

# Build Sankey links
sources = cluster_level["human_labelled"].map(node_index).tolist()
targets = cluster_level["cluster_name"].map(node_index).tolist()
values = cluster_level["cluster_size"].tolist()

fig = go.Figure(
    data=[
        go.Sankey(
            arrangement="snap",
            node=dict(
                pad=18,
                thickness=18,
                line=dict(color="rgba(80,80,80,0.5)", width=0.5),
                label=all_nodes,
            ),
            link=dict(
                source=sources,
                target=targets,
                value=values,
            ),
        )
    ]
)

fig.update_layout(
    title={
        "text": "Theme to sub-theme Sankey plot of barrier clusters",
        "x": 0.5,
        "xanchor": "center",
        "y": 0.98,
        "yanchor": "top",
    },
    font_size=11,
    width=780,
    height=800,
    margin=dict(t=40, l=20, r=20, b=20),
)


fig.show()
html_path = Path(r"C:\Users\FEEL\Downloads\images\rag_pipeline\roles-enhanced\analysis\theme_subtheme_sankey.html")
fig.write_html(html_path)
print(f"Saved Sankey to: {html_path}")

pdf_path = Path(r"C:\Users\FEEL\Downloads\images\rag_pipeline\roles-enhanced\analysis\theme_subtheme_sankey.pdf")
fig.write_image(pdf_path)
print(f"Saved Sankey PDF to: {pdf_path}")



In [2]:
#cell 8: jaccard similarity index heatmaps comparing similar role variants, using granular cluster IDs only

# cell: Jaccard heatmaps using granular cluster IDs only
# same-role panels
# keep only:
# - within-family 1 vs 2
# - cross-family 1 vs 1
# - cross-family 2 vs 2

from pathlib import Path
import matplotlib
matplotlib.use("Agg")  # safe in notebook/local headless runs
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

DATA_PATH = Path("analysis/context_barrier_mention_hdbscan_gpt5named_human.csv")

ROLE_ORDER = ["generalist", "normative", "subject_matter"]
ROLE_TITLES = {
    "generalist": "Generalist variants",
    "normative": "Normative variants",
    "subject_matter": "SME variants",
}
FAMILY_TITLES = {
    "phi4": "Phi-4",
    "gemma3": "Gemma3:12B",
    "llama": "Llama-Pro",
    "mistral": "Mistral",
}

rows = pd.read_csv(DATA_PATH)
rows["cluster"] = pd.to_numeric(rows["cluster"], errors="coerce")
rows["variant_id"] = rows["variant_id"].fillna("").astype(str).str.strip()

# use only non-noise rows
rows = rows[(rows["cluster"] >= 0) & (rows["variant_id"] != "")].copy()

def parse_variant_meta(variant_id: str):
    variant_id = str(variant_id).strip()

    if variant_id.startswith("phi4-"):
        family = "phi4"
        rest = variant_id[len("phi4-"):]
    elif variant_id.startswith("gemma3-"):
        family = "gemma3"
        rest = variant_id[len("gemma3-"):]
    elif variant_id.startswith("llama-"):
        family = "llama"
        rest = variant_id[len("llama-"):]
    elif variant_id.startswith("mistral-"):
        family = "mistral"
        rest = variant_id[len("mistral-"):]
    else:
        raise ValueError(f"Unrecognized variant_id: {variant_id}")

    version = 2 if rest.endswith("2") else 1
    role = rest[:-1] if version == 2 else rest
    return family, role, version

def variant_pretty(variant_id: str):
    family, _, version = parse_variant_meta(variant_id)
    return f"{FAMILY_TITLES[family]} V{version}"

variant_sets = {
    variant: set(sub["cluster"].astype(int).tolist())
    for variant, sub in rows.groupby("variant_id")
}

def jaccard_score(v1, v2):
    s1 = variant_sets.get(v1, set())
    s2 = variant_sets.get(v2, set())
    union = s1 | s2
    inter = s1 & s2
    return len(inter) / len(union) if union else 0.0

def keep_cell(v1, v2):
    if v1 == v2:
        return False

    family1, role1, version1 = parse_variant_meta(v1)
    family2, role2, version2 = parse_variant_meta(v2)

    if role1 != role2:
        return False

    # within-family profile comparison: 1 vs 2
    if family1 == family2 and version1 != version2:
        return True

    # cross-family direct matches only: 1 vs 1, 2 vs 2
    if family1 != family2 and version1 == version2:
        return True

    return False

family_order = {"phi4": 0, "gemma3": 1, "llama": 2, "mistral": 3}

role_groups = {}
for role in ROLE_ORDER:
    variants = [v for v in sorted(rows["variant_id"].unique()) if parse_variant_meta(v)[1] == role]
    variants.sort(key=lambda v: (family_order[parse_variant_meta(v)[0]], parse_variant_meta(v)[2], v))
    role_groups[role] = variants

fig, axes = plt.subplots(1, 3, figsize=(18, 6), constrained_layout=True)

cmap = plt.get_cmap("OrRd").copy()
cmap.set_bad("#f3f3f3")

for ax, role in zip(axes, ROLE_ORDER):
    variants = role_groups[role]
    pretty_labels = [variant_pretty(v) for v in variants]

    panel = pd.DataFrame(index=pretty_labels, columns=pretty_labels, dtype=float)
    mask = pd.DataFrame(True, index=pretty_labels, columns=pretty_labels)

    for v1 in variants:
        for v2 in variants:
            label1 = variant_pretty(v1)
            label2 = variant_pretty(v2)
            panel.loc[label1, label2] = jaccard_score(v1, v2)
            if keep_cell(v1, v2):
                mask.loc[label1, label2] = False

    data = panel.to_numpy(dtype=float)
    masked = data.copy()
    masked[mask.to_numpy(dtype=bool)] = np.nan

    im = ax.imshow(masked, cmap=cmap, vmin=0, vmax=1)
    ax.set_aspect("equal")

    if role == ROLE_ORDER[-1]:
        cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.set_label("Jaccard Similarity Index")

    for i in range(masked.shape[0]):
        for j in range(masked.shape[1]):
            if not np.isnan(masked[i, j]):
                ax.text(j, i, f"{masked[i, j]:.2f}", ha="center", va="center", fontsize=9)

    ax.set_title(ROLE_TITLES[role], fontsize=12)
    ax.set_xlabel("Variants")
    ax.set_ylabel("Variants" if role == ROLE_ORDER[0] else "")

    ax.set_xticks(range(len(pretty_labels)))
    ax.set_yticks(range(len(pretty_labels)))
    ax.set_xticklabels(pretty_labels, rotation=30, ha="right", rotation_mode="anchor")

    if role == ROLE_ORDER[0]:
        ax.set_yticklabels(pretty_labels, rotation=0)
    else:
        ax.set_yticklabels([])
        ax.tick_params(axis="y", length=0)

    # grid lines between cells
    ax.set_xticks([x - 0.5 for x in range(1, len(pretty_labels))], minor=True)
    ax.set_yticks([y - 0.5 for y in range(1, len(pretty_labels))], minor=True)
    ax.grid(which="minor", color="#9e9e9e", linestyle="-", linewidth=1)
    ax.tick_params(which="minor", bottom=False, left=False)

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(1.2)
        spine.set_edgecolor("#9e9e9e")

fig.suptitle("Jaccard similarity index heatmaps", fontsize=14)

save_path_png = Path("analysis/jaccard_overlap_heatmap_3panel_same_role_granular.png")
save_path_pdf = Path("analysis/jaccard_overlap_heatmap_3panel_same_role_granular.pdf")

plt.savefig(save_path_png, dpi=300, bbox_inches="tight")
plt.savefig(save_path_pdf, bbox_inches="tight")
plt.show()

print(f"Saved PNG: {save_path_png}")
print(f"Saved PDF: {save_path_pdf}")


Saved PNG: analysis\jaccard_overlap_heatmap_3panel_same_role_granular.png
Saved PDF: analysis\jaccard_overlap_heatmap_3panel_same_role_granular.pdf


C:\Users\FEEL\AppData\Local\Temp\ipykernel_19664\4034465528.py:173: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
